### 189. Rotate Array
Given an integer array `nums`, rotate the array to the right by `k` steps, where `k` is non-negative.

**Example 1:**
```shell
Input: nums = [1,2,3,4,5,6,7], k = 3
Output: [5,6,7,1,2,3,4]
```
**Explanation:**
```shell
rotate 1 steps to the right: [7,1,2,3,4,5,6]
rotate 2 steps to the right: [6,7,1,2,3,4,5]
rotate 3 steps to the right: [5,6,7,1,2,3,4]
```
**Example 2:**
```shell
Input: nums = [-1,-100,3,99], k = 2
Output: [3,99,-1,-100]
```
**Explanation:**
```shell
rotate 1 steps to the right: [99,-1,-100,3]
rotate 2 steps to the right: [3,99,-1,-100]
```
**Constraints:**

- `1 <= nums.length <= 105`
- `-231 <= nums[i] <= 231 - 1`
- `0 <= k <= 105`

**Follow up:**

- Try to come up with as many solutions as you can. There are at least **three** different ways to solve this problem.
- Could you do it in-place with $O(1)$ extra space?

### Reasoning
#### Step 1 - Gather requirements
I'm asked to rotate the given array to the right with `k` steps. The steps defined by `k`, which is **non-negative** integer number.

- Is the input sorted? What order if yes?
- Can the input be empty or null?
- Are there duplicates, negatives and zeros?
- Should the output be ordered?
- What are the bounds of input size?

**1. Is the input sorted?** No. The order is arbitrary.

**2. Can the input be empty or null?** No. `nums` always has at least 1 element.

**3. Duplicates, negatives, zeros?** Yes, all three can appear. Every value fits in a 32-bit signed int, from -2³¹ to 2³¹-1.

**4. Should the output be ordered?** For this problem, the order is the answer. **The result must be exactly the original array shifted right by `k`**. A sharper version of this question is: _"Do I return a new array, or modify the input?"_ Here's that answer:

- You must modify `nums` **in place**.
- The function returns nothing (`None`).
- Follow-up an interviewer will likely ask: can you do it with **$O(1)$ extra space**?

**5. Bounds on input size?**

- `1 <= len(nums) <= 10^5`
- `0 <= k <= 10^5`

Compare the two bounds carefully. They're independent, and that matters.

### Step 2 - Exampes + Edge Cases

```shell
nums = [1, 2, 3, 4, 5, 6, 7], k = 3
Output: [5, 6, 7, 1, 2, 3, 4]
1. [7, 1, 2, 3, 4, 5, 6]
2. [6, 7, 1, 2, 3, 4, 5]
3. [5, 6, 7, 1, 2, 3, 4]
```
```shell
nums = [-1, -100, 3, 99], k = 2
Output: [3, 99, -1, -100]
1. [99, -1, -100, 3]
2. [3, 99, -1, -100]
```

```shell
nums = [1, 2, 3], k = 4
Output: [4]
1. [3, 1, 2]
2. [2, 3, 1]
3. [1, 2, 3]  when k = n the output array is the same
4. [3, 1, 2]  k % n now gives number of rotates that change array
```
```shell
nums = [1, 2, 3, 4], k = 0
Output: [1, 2, 3, 4]
```

In [2]:
nums = [1, 2, 3]  # len(3)
k = 4  # rotate 4 times
len(nums) % k

3

> **Property 1**: `k = n` brings the array back to where it started is the **key insight** here.

> **Property 2**: every full lap of `n` rotations **cancels out**, and only the leftover rotations matter.

`k` gives the number of rotation to be done. We know if `n = k` makes a full cycle and ends up at the initial state. Once `k` starts to grow over `n`, we can say that another cycle starts **from scratch**. So, we can use modulo `%` operation, and see how much `n` goes into `k`, this will cancel full cycles out and leave only the number of rotations to be done. It is `k % n`. In this example `4 % 3 = 1`, one rotate to the right.

> `k % n` removes the **full cycles** and leaves only the rotations that actually change the array. It also handles `k = 0` safely, because the divisor is now `n`, which is always at least 1.

### Step 3 - Brute Force - `k % n` + `pop` + `insert`

Using `k mod n` instead of "n times" matters. When `k` is a multiple of `n`, this code does zero work.

In [5]:
def brute(nums: list[int], k: int) -> None:
    # brute force - k % n + .pop() + .insert()
    # setup
    # set the number of rotation times: rot_times = k % len(nums)
    # scan input - for loop - full pass

    # 1. pop and save the last item: last = nums.pop()
    # 2. insert the last item at the head of the array

    rot_times = k % len(nums)

    for _ in range(rot_times):
        last = nums.pop()
        nums.insert(0, last)

In [ ]:
# brute([1, 2, 3], 4)  # [3, 1, 2]

`pop()` is $O(1)$ and `insert(0, ...)` is $O(n)$, so each rotation costs $O(n)$.

- **Time**: $O(n\space \times (k\space \text mod\space n))$, which is $O(n²)$ in the worst case. The loop runs `k` mod `n` times, at most `n-1`, and each `insert(0)` shifts all `n` elements.
- **Space**: $O(1)$. Only one saved element; the rotation happens in place.

**Find the redundant work**

In [7]:
(0 + 3) % 7

3

In [6]:
brute([1, 2, 3, 4, 5, 6, 7], 3)  # [5, 6, 7, 1, 2, 3, 4]

Element `1` gets shifted 3 times, one per rotation.
If I already knew where it belonged, I could write it straight to index 3.

> **Redundant Work**: The Brute force moves every element `k` times, one step per rotation, when one direct move per element would do.

> **Property**: elements `1` starts at index 0, gets rotated 3 times, and lands at index 3. Let's turn it into a rule.

In the same example, element `5` starts at index 4 and ends at index 0:
```shell
index:   0  1  2  3  4  5  6
before: [1, 2, 3, 4, 5, 6, 7]
after:  [5, 6, 7, 1, 2, 3, 4]
```
![Visual](../../images/189rotate_array_formula.png)